[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/beltrami_fields/beltrami_fields.ipynb)

# Beltrami Fields

This example computes Beltrami fields, the force-free states of a magnetic field, in which the electric current follows the field, so that the field exerts no force on the current that carries it. Magnetic fields keep their linkage and twist even as their finer structure fades, and the force-free state of least energy is where a field of given linkage settles.

The field is static, so the example lives in one instant of spacetime, `x+y+z+t-`, with the direction of time `time` of an observer at rest with the torus. The magnetic field is modelled as a bivector, a plane of space, and an oriented patch reads its flux as `field & patch`. Positions and patches are directions and patches of space; a hat function is constant in time, so its gradient is the volume its opposite face sweeps through time, read against the prism the tetrahedron sweeps, `volume & time`, and the Whitney forms built from these gradients need no metric. The metric enters the energy alone, through the vacuum `Field.dual()`, which takes the field to the excitation it causes; the helicity wedges the potential with its curl and pairs the result with the signed volume, with no metric at all. The adjugate of the Whitney reconstruction carries both readings back to the same edge amplitudes. The Whitney forms of a single tetrahedron, and the volume pairing of its edge forms with their curls that links only opposite edges, are worked out in the plain algebra of space in [Whitney Forms](../../surfaces/whitney_forms/whitney_forms.ipynb).

In vector-calculus notation, a Beltrami field satisfies $\nabla\times\mathbf B = \lambda\mathbf B$; its energy is $E = \frac{1}{2}\int|\mathbf B|^2\,dV$ and its helicity $H = \int\mathbf A\cdot\mathbf B\,dV$ with $\mathbf B = \nabla\times\mathbf A$, and the Beltrami fields are the stationary points of $E - \frac{\lambda}{2}H$.


In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np
from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import spdiag
from examples.electromagnetism.beltrami_fields import render
from examples.electromagnetism.beltrami_fields.plumbing import as_scalar, flux_preserving, solid_torus
from examples.electromagnetism.beltrami_fields.numerical import coexact_helicity_eigh
from examples.util.animation import save_figure, show_animation

ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()          # Scalar+
Plane = ga.gatype.vector()           # Vector+
# The magnetic field, a plane of space.
Field = ga.gatype.bivector().derive.within("x y z")  # Bivector+
Curl = ga.gatype((Field, Scalar))

# The direction in time of an observer at rest with the torus, which pairs to one with `mv.t`.
time = -mv.t.dual()                  # [] Point-

## Beltrami modes in a torus

Energy reads the field through the vacuum, while helicity reads its linkage through signed volume. Both are forms on the edge amplitudes, and a Beltrami mode makes their responses proportional. A reflected field, read on the same positively oriented tetrahedra, keeps its energy and reverses its helicity, so the modes come in pairs of opposite handedness.

In [ ]:
major_radius = 1.0
minor_radius = 0.42
n_major = 16
n_minor = 8
shown_indices = np.array([-1, -4])

In [ ]:
mesh, vertices = solid_torus(major_radius, minor_radius, n_major, n_minor)  # Mesh, Point-[vertices]
corners = mesh.incidence[3, 0] * vertices                                  # [corners] Point-[tets]
centres = mesh.incidence[3, 0].mean(axis=0) * vertices                     # Point-[tets]
# Each tetrahedron's edges in its own vertex order; the first three run from its first corner.
edges = mesh.oriented[3, 1] * (mesh.d[0] * vertices)                       # [tet_edges] Point-[tets]
volumes = (edges[0] & edges[1] & edges[2]) / 6                             # Volume-[tets]
prisms = volumes & time                                                    # Scalar-[tets]
# Each face's patch, half the join of two of its sides, read as the face facing each corner, outward.
sides = mesh.boundary[2, 1] * (mesh.d[0] * vertices)                       # [3] Point-[faces]
face_patches = mesh.boundary[3, 2] * ((sides[0] & sides[1]) / 2)           # [corners] Patch-[tets]

# The corner planes read each hat's rise across its opposite face.
corner_gradients = (face_patches & time) / (3 * prisms)                    # [corners] Plane+[tets]

# At each tetrahedron centroid the four barycentric hats equal 1/4, so the edge 1-form
# reduces to one quarter of the difference of its endpoints' corner gradients.
tails, heads = mesh.corners(3, 1).T
centroid_1forms = (corner_gradients[heads] - corner_gradients[tails]) / 4  # [tet_edges] Plane+[tets]
# A tet's edges run from its lower local corner to its higher, the mesh's edges from the lower vertex to the
# higher: `mesh.oriented[3, 1]` reads each mesh edge with the sign of their agreement.
whitney1 = (spdiag(centroid_1forms * Scalar) * mesh.oriented[3, 1]).sum(axis=0)  # (Plane+ <- Scalar+)[tets <- edges]

# The wedge of endpoint gradients reads the circulation through a patch.
curls = 2 * (corner_gradients[tails] ^ corner_gradients[heads])  # [tet_edges] Field+[tets]
curl = (spdiag(curls * Scalar) * mesh.oriented[3, 1]).sum(axis=0)          # (Field+ <- Scalar+)[tets <- edges]

In [ ]:
# Magnetic energy pulls the vacuum, which takes each tetrahedron's field to the excitation it causes
# weighted by its volume, back through the curl; magnetic helicity pulls the metric-free volume
# pairing back through the centroid 1-form and the curl.
vacuum = spdiag(prisms.norm() * Field.dual())                              # (Patch- <- Field+)[tets <- tets]
stiffness = curl.adjugate()(vacuum(curl))                                  # (Amount- <- Scalar+)[edges <- edges]
volume_weights = spdiag(Plane ^ volumes)                                   # (Patch- <- Plane+)[tets <- tets]
raw_helicity = curl.adjugate()(volume_weights(whitney1))                   # (Amount- <- Scalar+)[edges <- edges]
helicity = (raw_helicity + raw_helicity.adjugate()) / 2                    # (Amount- <- Scalar+)[edges <- edges]

The conducting wall carries no outward flux. A circulation around the tube remains free, allowing magnetic flux around the torus.

In [ ]:
free = flux_preserving(mesh, n_major, n_minor)                             # (Scalar+ <- Scalar+)[edges <- free]
# Directions with zero magnetic energy change the potential without changing its field.
free_energy = free.adjugate()(stiffness(free))                             # (Amount- <- Scalar+)[free <- free]
free_helicity = free.adjugate()(helicity(free))                            # (Amount- <- Scalar+)[free <- free]
inverse_eigenvalues, free_modes, _ = coexact_helicity_eigh(free_energy, free_helicity)  # [modes] Scalar+, [modes] Scalar+[free], int
modes = free(free_modes)                                                   # [modes] Scalar+[edges]
shown_modes = modes[shown_indices]                                         # [shown] Scalar+[edges]
shown_fields = curl(shown_modes)                                           # [shown] Field+[tets]

save_figure(
    render.draw_beltrami_eigenmodes(mesh, vertices, centres, volumes, inverse_eigenvalues, shown_fields, shown_indices),
    "beltrami_eigenmodes",
);

The spectrum of inverse eigenvalues, each mode's helicity per unit squared energy norm, pairs positive and negative helicities. The two marked modes appear beside it: the lowest positive-helicity mode and a higher mode with more internal structure. Tubes follow a smoothed, wall-confined reconstruction of the tetrahedral field; colours distinguish seed layers, and arrowheads indicate direction.

In the matrix notation of finite elements, the energy and helicity forms give $Sa=\lambda Ha$, with $S_{ee'}=\int dw_e\wedge\star dw_{e'}$ and $H_{ee'}=\frac12\int(w_e\wedge dw_{e'}+w_{e'}\wedge dw_e)$.

## Selective decay

Fine twists fade faster than broad circulation. The amplitudes below follow a prescribed damping law with a shared rescaling that holds total helicity fixed, illustrating the approach toward the lowest-energy mode carrying that linkage. The two views show the same field settling into a coherent helical structure.

This cell takes about ten seconds, mainly to draw the tubes.

In [ ]:
selected_indices = np.array([-1, -2, -4, -8, 3])
initial_weights = np.array([0.45, 0.70, 0.85, 0.85, 0.45])
n_frames = 46
relaxation_time = 0.025
frame_duration_ms = 40

In [ ]:
def selective_decay(initial: Scalar, eigenvalues: Scalar, times: np.ndarray) -> Iterator[Scalar]:
    """Yield selectively damped modal amplitudes rescaled to preserve their initial helicity."""
    initial_helicity = (initial.squared() / eigenvalues).sum(axis=0)          # [] Scalar+
    for elapsed in times:
        # Finer modes decay faster; the shared scale fixes the linkage of the whole field.
        damped = initial * (-eigenvalues.squared() * elapsed).exp()           # [selected] Scalar+
        helicity = (damped.squared() / eigenvalues).sum(axis=0)               # [] Scalar+
        yield damped * (initial_helicity / helicity).square_root()


def reconstruct(states: Iterable[Scalar], modes: Scalar, curl: Curl) -> Iterator[Field]:
    """Yield the reconstructed magnetic flux field for each modal state."""
    for amplitudes in states:
        cochain = (amplitudes * modes).sum(axis=0)                            # Scalar+[edges]
        yield curl(cochain)


selected_modes = modes[selected_indices]                                    # [selected] Scalar+[edges]
selected_eigenvalues = 1 / inverse_eigenvalues[selected_indices]            # [selected] Scalar+
initial_amplitudes = as_scalar(initial_weights)                             # [selected] Scalar+
times = np.linspace(0.0, relaxation_time, n_frames)
states = selective_decay(initial_amplitudes, selected_eigenvalues, times)
fields = reconstruct(states, selected_modes, curl)
show_animation(
    render.animate_relaxation(mesh, vertices, centres, volumes, fields),
    "beltrami_relaxation", frame_duration_ms, n_frames,
)

With the helicity held fixed, the energy falls toward the least that the helicity allows, that of the Taylor state, the lowest mode alone.

In modal notation, $E=\frac12\sum_k c_k^2$ and $H=\sum_k c_k^2/\lambda_k$. Multiplying $c_k(0)e^{-\lambda_k^2t}$ by a common factor fixes $H$; for this mode mixture, the energy approaches $\lambda_1 H/2$.

In [ ]:
# checks
assert (inverse_eigenvalues + inverse_eigenvalues[::-1]).norm().sum() < 1e-11
shown_eigenvalues = 1 / inverse_eigenvalues[shown_indices]
shown_energies = (stiffness(shown_modes) & shown_modes).sites.sum() / 2
shown_helicities = (helicity(shown_modes) & shown_modes).sites.sum()
shown_residuals = free.adjugate()(stiffness(shown_modes) - shown_eigenvalues * helicity(shown_modes))
assert (shown_energies - 0.5).norm().sum() < 1e-11
assert (shown_helicities - inverse_eigenvalues[shown_indices]).norm().sum() < 1e-11
assert shown_residuals.dual_inverse().norm().sites.sum().sum() < 1e-10

amplitude_frames = stack(list(selective_decay(initial_amplitudes, selected_eigenvalues, times)))
energy_frames = amplitude_frames.squared().sum(axis=1) / 2
helicity_frames = (amplitude_frames.squared() / selected_eigenvalues).sum(axis=1)
taylor_bound = helicity_frames[0] * selected_eigenvalues[0] / 2
assert (helicity_frames - helicity_frames[0]).norm().sum() < 1e-11
assert energy_frames[-1] > taylor_bound
assert (energy_frames[-1] - taylor_bound) / taylor_bound < 0.01